# Data Preprocessing with PySpark and MongoDB

This notebook connects to MongoDB, performs data preprocessing on a PySpark DataFrame, and explores key data features.

## MongoDB Configuration and PySpark Setup



In [ ]:
# MongoDB credentials
username = "guest"
password = "Guest123"
host = "cluster0.xpghy.mongodb.net"
database = "Lungcancer"
collection = "Patients"

# SETUP MONGODB CLIENT
from pymongo import MongoClient # import mongo client to connect
import urllib.parse
# assign credentials to variables
username = username
password = urllib.parse.quote(password)
host = host
url = "mongodb+srv://{}:{}@{}/?retryWrites=true&w=majority".format(username,
password, host)
# connect to the database
client = MongoClient(url)



## Loading Data from MongoDB to PySpark DataFrame

Load data from the MongoDB collection into a PySpark DataFrame for processing.

In [ ]:
# Read data from MongoDB into PySpark DataFrame
df_spark = spark.read.format('com.mongodb.spark.sql.DefaultSource') \
    .option('uri', f'{url}/{database}.{collection}') \
    .load()

# Show initial rows from the DataFrame
df_spark.show(5)


## Data Transformation

We will convert integer columns to binary values (0 or 1) for easier analysis.

In [ ]:
# Transform specific columns to binary values (0 or 1)
df_spark = df_spark.withColumn('SMOKING', when(df_spark.SMOKING == 1, 1).otherwise(0)) \
    .withColumn('YELLOW_FINGERS', when(df_spark.YELLOW_FINGERS == 1, 1).otherwise(0)) \
    .withColumn('ANXIETY', when(df_spark.ANXIETY == 1, 1).otherwise(0)) \
    .withColumn('FATIGUE', when(df_spark.FATIGUE == 1, 1).otherwise(0)) \
    .withColumn('WHEEZING', when(df_spark.WHEEZING == 1, 1).otherwise(0)) \
    .withColumn('ALCOHOL_CONSUMING', when(df_spark.ALCOHOL_CONSUMING == 1, 1).otherwise(0)) \
    .withColumn('COUGHING', when(df_spark.COUGHING == 1, 1).otherwise(0)) \
    .withColumn('SHORTNESS_OF_BREATH', when(df_spark.SHORTNESS_OF_BREATH == 1, 1).otherwise(0)) \
    .withColumn('SWALLOWING_DIFFICULTY', when(df_spark.SWALLOWING_DIFFICULTY == 1, 1).otherwise(0)) \
    .withColumn('CHEST_PAIN', when(df_spark.CHEST_PAIN == 1, 1).otherwise(0))

# Show transformed DataFrame
df_spark.show(5)


## Writing Transformed Data Back to MongoDB

Save the transformed DataFrame back into MongoDB.

In [ ]:
# Write transformed DataFrame back to MongoDB
df_spark.write \
    .format('mongo') \
    .mode('append') \
    .option('uri', url) \
    .option('database', database) \
    .option('collection', collection) \
    .save()


## Data Exploration and Analysis

We will explore schema, distribution of `AGE`, and the count of each `GENDER`.

In [ ]:
# Display schema of the DataFrame
df_spark.printSchema()


In [ ]:
# 1. Count occurrences of each value in the 'AGE' column
df_spark.groupBy('AGE').count().show(20)


In [ ]:
# 2. Calculate minimum and maximum age
from pyspark.sql import functions as F

min_age = df_spark.agg(F.min('AGE')).collect()[0][0]
max_age = df_spark.agg(F.max('AGE')).collect()[0][0]
print(f'Minimum Age: {min_age}, Maximum Age: {max_age}')


In [ ]:
# 3. Show unique ages in ascending order
df_spark.select('AGE').distinct().orderBy('AGE').show(20)


In [ ]:
# 4. Count occurrences of each gender
df_spark.groupBy('GENDER').count().show()